In [19]:
from scipy.special import kolmogorov
import numpy as np
import math

In [20]:
#Функция нормального распределения.
def F_norm(x, mean, sigma):
  return 0.5 * (1 + math.erf((x - mean) / (np.sqrt(2) * sigma)))


m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

N = 100

## a)

In [ ]:
F_emp = np.array([sum(m[:i]) for i in range(len(m) + 1)]) / N
F_th = np.arange(10) / 10

delta = np.sqrt(N) * np.max([max(np.abs(F_th[i] - F_emp[i]),
                                 np.abs(F_th[i] - F_emp[i + 1])) for i in range(F_th.size)])

p_value = kolmogorov(delta)


print(f"p-value Колмогорова: {p_value}")

p-value: 0.039681879538114355


## b)

In [22]:
# Количество повторений bootstrap
bootstrap_iterations = 50_000

In [23]:
# Полуинтервалы полной группы событий
segments = np.array([(-np.inf, 1)] + [(i, i + 1) for i in range(1, 9)] + [(9, np.inf)])

# Оригинальная выборка
sample = np.repeat(np.arange(len(m)), m)

# Оценка мат. ожидания
alpha = np.mean(sample)

# Оценка корня дисперсии (несмещенная)
sigma = np.sqrt(np.var(sample) * N / (N - 1))


def F_norm_wave(x): return F_norm(x, alpha, sigma) 

In [ ]:
bootstrap_delta = []

x = np.arange(10)
delta_wave = np.sqrt(N) * np.max([max(np.abs(F_norm_wave(x[i]) - F_emp[i]),
                                      np.abs(F_norm_wave(x[i]) - F_emp[i + 1])) for i in x])

for _ in range(bootstrap_iterations):
  # Случайная выборка, соотв норм. распределению с параметрами из О.М.П.Г.
  random_sample = np.array(sorted(np.random.normal(alpha, sigma, N)))

  # Оценка мат. ожидания по случайной подвыборке (bootstrap)
  alpha_bootstrap = random_sample.mean()

  # Оценка корня дисперсии по случайной подвыборке (bootstrap)
  sigma_bootstrap = np.sqrt(random_sample.var() * N / (N - 1))

  F_bootstrap_emp = [i / N for i in range(N + 1)]

  # Функция нормального распределения по случайной подвыборке (bootstrap)
  def F_bootstrap_wave(j):
    return F_norm(random_sample[j], alpha_bootstrap, sigma_bootstrap)

  sup = np.max([max(np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j]),
                    np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j + 1]))
                for j in range(len(random_sample))])

  bootstrap_delta.append(np.sqrt(N) * sup)

bootstrap_delta = np.array(bootstrap_delta)

p_value = len(bootstrap_delta[bootstrap_delta >= delta_wave]) / bootstrap_iterations




In [27]:
print(f"p-value Колмогорова: {p_value}")

p-value Колмогорова: 0.01484


In [29]:
# Вероятность каждого события
P = [F_norm_wave(i[1]) - F_norm_wave(i[0]) for i in segments]

delta = sum((N * P[i] - m[i]) ** 2 / (N * P[i]) for i in range(10))

print(f"Δ = {delta}")

Δ = 16.87106704806872
